# Laya FP16: faster inference with unchanged weights

Uses **`vtava/Laya-1.2-speedup-standalone-fp16`**, pinned to model revision
`2891f7312d8e7c65b43a6e04c29cc226b06b4fac`. No retraining or upstream base model is needed.

The tournament compares original FP16, device-side masks + gather-first, a marker-only
final head layer, `torch.compile`, and CUDA graphs. Optional FlashAttention-2 requires
compatible hardware/software. Every candidate must preserve all tested decisions and
stay within 0.01 maximum probability drift. The original remains eligible if faster.

**Runtime → Change runtime type → GPU**, then run all cells. Compilation can take several
minutes per shape; its startup cost is reported separately. CPU checks do not establish
GPU speed. Synthetic parity is not held-out task accuracy. No checkpoint is uploaded.

The 28 encoder layers, weights, windows, token order, and head landmarks are preserved.
Do not bucket/rep­ad inputs without a new quality check: this model selects landmarks
using the padded sequence length.

In [ ]:
# Install the checkpoint's pinned Transformers version. Keep Colab's CUDA PyTorch.
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json, subprocess, sys, tempfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first."
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
# Fresh clone avoids overwriting any existing notebook/project work.
REPO = Path(tempfile.mkdtemp(prefix="laya-fast-")) / "TinyCeNN-LM"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/laya-fp16-fast-inference",
                "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

In [ ]:
# Benchmark settings. Start with 128 and 512; add 1024/2048 for your workload.
LENGTHS = [128, 512]
BATCH_SIZES = [1, 4]
TRY_COMPILE = True
COMPILE_MODE = "reduce-overhead"  # "max-autotune" trades longer startup for kernel tuning
TRY_CUDA_GRAPH = True
TRY_FLASH_ATTENTION = False      # Optional; T4 does not support this FA2 path
INSTALL_FLASH_ATTENTION = False  # Enable explicitly only on a supported GPU
if INSTALL_FLASH_ATTENTION:
    assert torch.cuda.get_device_capability(0)[0] >= 8, "Use Ampere or newer, not T4."
    subprocess.run([sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"], check=True)
    TRY_FLASH_ATTENTION = True
REPORT = Path("/content/laya_fp16_benchmark.json")

In [ ]:
# Runs in a fresh process: compile/capture failures are logged, CUDA corruption stops.
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_laya_fp16.py"),
           "--device", "cuda", "--lengths", *map(str, LENGTHS),
           "--batch-sizes", *map(str, BATCH_SIZES), "--output", str(REPORT)]
if TRY_COMPILE: command += ["--compile", "--compile-mode", COMPILE_MODE]
if TRY_CUDA_GRAPH: command += ["--cuda-graph"]
if TRY_FLASH_ATTENTION: command += ["--flash-attention"]
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Markdown, display
report = json.loads(REPORT.read_text())
rows = ["| Batch | Tokens | Fastest accepted path | Median ms | Speedup |",
        "|---:|---:|---|---:|---:|"]
for w in report["winners"]:
    rows.append(f"| {w['batch_size']} | {w['sequence_length']} | {w['candidate']} | {w['median_ms']:.3f} | {w['speedup_vs_original']:.3f}× |")
display(Markdown("\n".join(rows)))
print("End-to-end API timings:", json.dumps(report["end_to_end"], indent=2))
print("Skipped candidates:", report["skipped"])
print("Each result includes startup cost, all parity audits, and individual timing rounds.")

## Use the optimized path on your own input

Edit `STATE` and `QUESTIONS`. This example checks the eager optimized path against
the original before using it. For deployment, validate representative states and
question types, then choose the measured fastest path for your workload. CUDA graphs
require identical shapes/dtypes/devices; different contents and padding masks are allowed.
The benchmark measures model latency with GPU-resident inputs separately from API latency.

In [ ]:
from huggingface_hub import snapshot_download
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO / "standalone/laya_1p2_speedup"))
from benchmark_laya_fp16 import MODEL_ID, MODEL_REVISION, QUESTIONS, load_runtime, parity
from fast_inference import optimize_model, PreparedChoices, CUDAGraphRunner, inference_context
folder = snapshot_download(MODEL_ID, revision=MODEL_REVISION)
runtime = load_runtime(folder)
original, tokenizer = runtime.load_model(folder, device="cuda")
fast = optimize_model(original)
prepared = PreparedChoices(runtime, tokenizer, QUESTIONS)
STATE = {"road_direction": "left", "speed": 7.0}
batch = prepared.batch(STATE, device="cuda")
with torch.inference_mode(), inference_context(original):
    reference = original(**batch)[0].clone()
    result = fast(**batch)[0].clone()
audit = parity(reference, result, batch["marker_mask"])
print("Your input parity:", audit)
selected = fast if audit["passed"] else original
print(prepared.predict(selected, STATE))

# Optional fixed-shape serving after validating graph output as well:
# graph = CUDAGraphRunner(fast, batch)
# with torch.inference_mode():
#     logits, _ = graph(**batch)
#     assert parity(reference, logits, batch["marker_mask"])["passed"]
#     print(prepared.decode(logits, batch["marker_mask"]))

In [ ]:
# Download the complete measured report; no Hugging Face write or login required.
from google.colab import files
files.download(str(REPORT))